# Phase 9 · Section 1 — Admin Dashboard

### Objective

Phase 9 Section 1 upgrades the portfolio from a managed content system into a **fully authenticated, production-grade admin platform** — one that supports ongoing content management, tag organisation, and project control entirely through a secure UI.

While Phase 8 introduced an admin panel as an extension of the copywriting workflow, this section treats the admin dashboard as a **first-class engineering deliverable** in its own right.

---

## Status

**Sections 1.1 and 1.2 were completed ahead of schedule during Phase 8 Section 1**, where the admin panel was built from scratch as an extension beyond the original spec. The implementation exceeded the requirements defined here.

**Section 1.3 (Tag and Category Management)** is the only outstanding item. Tags are currently editable as comma-separated text on the project edit form, but there is no dedicated management interface.

| Section | Status | Delivered in |
|---|---|---|
| 1.1 — Login and Session Auth | ✅ Complete | Phase 8 Section 1 |
| 1.2 — Project CRUD | ✅ Complete | Phase 8 Section 1 |
| 1.3 — Tag and Category Management | ❌ Not built | — |

---

## What Was Built (Phase 8 Section 1)

The following was implemented beyond the original spec and covers all of 1.1 and 1.2:

**Authentication**
- Password-protected login with session auth (`_require_admin` decorator)
- All admin routes protected — decorator applied to every route in `app/admin/routes.py`
- `ADMIN_PASSWORD` stored in environment variable, never hardcoded
- Login, logout, and session clear all working

**Project CRUD**
- Full create, edit, delete workflow via form UI
- All project fields covered: title, slug, category, tags, tech stack, descriptions, problem/solution/challenges/results, repo URL, live URL, demo URL, featured flag, date
- Media manager: card image, screenshot, and video upload to local static storage
- Delete requires a POST confirmation — no single-click delete

**Dashboard**
- Live stats: project count, featured count, message count
- Project list with search by title and filter by category

**Messages**
- Contact form submissions viewable and deletable through the admin UI

**Backup System** *(beyond spec)*
- On-demand timestamped JSON snapshots saved to `data/backup/`
- Auto-retention of 3 most recent backups
- Restore via admin UI with automatic pre-restore safety snapshot
- Upsert by slug for safe incremental restores

---

## Actual Files (as built)

```bash
app/admin/__init__.py           # Blueprint registration
app/admin/routes.py             # All routes — login, logout, dashboard, CRUD, media, messages, backups
app/admin/forms.py              # ProjectForm definition
app/models/models.py            # Project and ContactMessage models (single file, not split)
app/templates/admin/
    login.html
    dashboard.html
    projects_list.html
    project_form.html           # Shared create/edit form
    media_manager.html
.env                            # ADMIN_PASSWORD (gitignored)
```

> Note: the original spec referenced `app/admin/decorators.py`, `app/models/category.py`, `app/models/tag.py`, and `app/templates/admin/taxonomy/`. None of these were created — the decorator lives inline in `routes.py`, and categories/tags are string fields on the Project model.

---

## Prerequisites

The following components must already exist:

* Fully deployed application with public URL
* Projects stored in a database (not flat files)
* Existing admin routes or blueprints in place
* Stable production environment

These were implemented in:

* **Phase 7 · Section 2 — Hosting and Deployment**
* **Phase 8 · Section 1 — Copywriting and Narrative**
* **Phase 8 · Section 1v2 — Admin Panel (initial build)**

---

# 1.1 — Login and Session-Based Authentication ✅

## Status: Complete — delivered in Phase 8 Section 1

All requirements met. Summary of what was built:

- Single admin account authenticated via `ADMIN_PASSWORD` environment variable
- Session set on successful login, cleared on logout
- `_require_admin` decorator applied to every admin route
- Unauthenticated requests to any `/admin/*` route redirect to `/admin/login`
- No credentials hardcoded anywhere in source

---

## Auth Strategy

The auth system:

* accepts a password via a login form
* validates against `ADMIN_PASSWORD` stored in environment config
* stores authenticated state in a server-side session
* redirects unauthenticated requests to the login page
* clears the session on logout

---

## Login Required Decorator

Implemented inline in `app/admin/routes.py`:

```python
from functools import wraps
from flask import session, redirect, url_for

def _require_admin(f):
    @wraps(f)
    def decorated(*args, **kwargs):
        if not session.get('admin_logged_in'):
            return redirect(url_for('admin.login'))
        return f(*args, **kwargs)
    return decorated
```

Applied to every admin route:

```python
@admin_bp.get('/dashboard')
@_require_admin
def dashboard():
    ...
```

---

## Validation Checklist

- [x] Visiting any `/admin/*` route while logged out redirects to `/admin/login`
- [x] Correct credentials create a session and redirect to the dashboard
- [x] Incorrect credentials return to the login page with an error message
- [x] Logout clears the session and redirects to the login page
- [x] No credentials are hardcoded in any Python file or template

---

# 1.2 — Add, Edit, and Remove Projects ✅

## Status: Complete — delivered in Phase 8 Section 1

All requirements met. Full CRUD implemented across all project fields.

---

## What Was Built

**Routes in `app/admin/routes.py`:**

```bash
GET  /admin/projects                  # Project list with search and category filter
GET  /admin/projects/new              # New project form
POST /admin/projects/new              # Create project
GET  /admin/projects/<slug>           # Edit project form (pre-populated)
POST /admin/projects/<slug>           # Update project
POST /admin/projects/<slug>/delete    # Delete project (POST confirmation required)
GET  /admin/projects/<slug>/media     # Media manager
POST /admin/projects/<slug>/media/card        # Upload card image
POST /admin/projects/<slug>/media/screenshot  # Upload screenshot
POST /admin/projects/<slug>/media/video       # Upload video
POST /admin/projects/<slug>/media/delete      # Delete a media file
```

**Form fields covered:**
- title, slug, primary_category, featured, date
- short_description, full_description
- problem, solution, challenges, results
- tags, tech_stack (comma-separated)
- repo_url, live_url, demo_url

---

## Validation Checklist

- [x] A new project created through the form appears immediately on the public projects page
- [x] Editing a project updates all fields in the database and reflects on the public site
- [x] Deleting a project requires a POST confirmation and removes it from the database
- [x] File uploads save to the correct static directory and the path is stored in the database
- [x] Media manager handles card images, screenshots, and videos independently

---

# 1.3 — Manage Tags and Categories ❌

## Status: Not built

Tags are currently editable as comma-separated text in the project edit form. There is no dedicated interface for creating, renaming, or deleting tags across projects. Categories are string fields with no management UI.

---

## What Is Still Needed

### Tag Management Interface

A dedicated admin page that:

* lists all tags currently in use with a count of how many projects use each
* allows renaming a tag — updates the tag across all projects that use it
* allows deleting a tag — with confirmation and a warning if projects still use it

### Category Management

Categories (`web`, `data`, `software`) are currently hardcoded in templates and form choices. If categories need to change:

* a management interface would allow adding or renaming categories
* deleting a category should be blocked or redirected if projects are assigned to it

> Note: given only 3 categories exist and they are unlikely to change, category management may not be worth building. Tag management is the higher-value item.

---

## Implementation Approach (when built)

Tags are stored as JSON on the Project model. A tag management page would need to:

1. Query all projects and extract a flattened, deduplicated tag list with counts
2. For rename: iterate all projects, update the tag in each project's tag list, save
3. For delete: same iteration, remove the tag from each project's list

No separate Tag model is required — this can be done purely through the existing Project model.

```python
# Example: rename a tag across all projects
old_tag = "web-app"
new_tag = "web-application"

for project in Project.query.all():
    if old_tag in project.tags:
        updated = [new_tag if t == old_tag else t for t in project.tags]
        project.tags = updated

db.session.commit()
```

---

## Validation Checklist (when built)

- [ ] All tags visible in management interface with project counts
- [ ] Renaming a tag updates it across all affected projects
- [ ] Deleting a tag requires confirmation and shows affected project count
- [ ] New tags added via project edit form appear in the management interface automatically

---

# Admin UI Audit

## Current State

The admin UI is functional and consistent. All pages share the same layout, flash messages are used for success and error states, and the admin interface is visually distinct from the public site.

Outstanding gap: no taxonomy management pages exist yet (`/admin/tags`, `/admin/categories`).

---

## Audit Checklist

**Authentication**
- [x] Every admin route is protected by `_require_admin`
- [x] Login page handles incorrect credentials gracefully
- [x] Logout is accessible from every admin page

**Project CRUD**
- [x] Project list shows all projects with key metadata
- [x] Create and edit forms include every field in the data model
- [x] Delete flow is protected by a confirmation step
- [x] Media uploads save correctly and display on the public site

**Tag and Category Management**
- [ ] Dedicated tag management interface
- [ ] Dedicated category management interface
- [ ] Deletion guards in place

**Global Admin Consistency**
- [x] Navigation consistent across all admin pages
- [x] Flash messages used for all success and error states
- [x] Admin UI visually distinct from the public site

---

# Result

Sections 1.1 and 1.2 are production-grade and fully operational. The admin dashboard provides:

* hardened session-based authentication protecting all admin routes
* a complete project CRUD interface covering every field in the data model
* media management for card images, screenshots, and videos
* a backup and restore system not originally in scope
* a coherent, navigable admin UI clearly separated from the public site

The one outstanding item is the tag management interface (1.3). Everything else exceeds the original spec.

---

# Final Position

At this point, your project includes:

* **Phase 7 · Section 2 — Hosting and Deployment**
* **Phase 7 · Section 3 — Custom Domain and HTTPS Configuration**
* **Phase 7 · Section 4 — Analytics and SEO Foundations**
* **Phase 8 · Section 1 — Copywriting and Narrative**
* **Phase 8 · Section 2 — Proof Signals**
* **Phase 9 · Section 1 — Admin Dashboard** *(1.1 and 1.2 complete, 1.3 outstanding)*

---